In [28]:

"""
generate_data.py — Synthetic Sales Data Generator
Sales Performance Dashboard | Project 01
Generates ~12,000 rows of realistic retail sales data with intentional quality issues.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)

# ── Configuration ──
NUM_ROWS = 12000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

REGIONS = ["Northeast", "Southeast", "Midwest", "West", "Southwest"]
REGION_WEIGHTS = [0.25, 0.20, 0.20, 0.22, 0.13]

CATEGORIES = {
    "Electronics":    {"price_range": (49.99, 999.99), "margin_range": (0.08, 0.25)},
    "Apparel":        {"price_range": (14.99, 199.99), "margin_range": (0.35, 0.60)},
    "Home & Garden":  {"price_range": (9.99, 499.99),  "margin_range": (0.25, 0.45)},
    "Sports":         {"price_range": (19.99, 349.99), "margin_range": (0.20, 0.40)},
    "Office Supplies": {"price_range": (4.99, 149.99), "margin_range": (0.30, 0.55)},
}

PRODUCTS = {
    "Electronics":     ["Wireless Headphones", "Bluetooth Speaker", "Tablet", "Smart Watch", "Webcam", "Portable Charger"],
    "Apparel":         ["Running Shoes", "Winter Jacket", "Polo Shirt", "Joggers", "Baseball Cap", "Sunglasses"],
    "Home & Garden":   ["Air Purifier", "Plant Pot Set", "LED Desk Lamp", "Throw Blanket", "Tool Kit", "Candle Set"],
    "Sports":          ["Yoga Mat", "Dumbbells", "Resistance Bands", "Jump Rope", "Foam Roller", "Water Bottle"],
    "Office Supplies": ["Notebook Set", "Ergonomic Mouse", "Desk Organizer", "Whiteboard", "Planner", "Pen Set"],
}

CUSTOMER_SEGMENTS = ["Consumer", "Corporate", "Small Business"]
SEGMENT_WEIGHTS = [0.50, 0.30, 0.20]

SALES_REPS = [
    "Marcus Thompson", "Priya Sharma", "James O'Brien", "Sofia Rodriguez",
    "David Kim", "Aisha Patel", "Ryan Mitchell", "Chen Wei",
    "Olivia Foster", "Kwame Asante", "Emily Nakamura", "Carlos Mendez",
]

# Assign reps to regions
REP_REGIONS = {}
reps_per_region = len(SALES_REPS) // len(REGIONS)
for i, rep in enumerate(SALES_REPS):
    region_idx = min(i // reps_per_region, len(REGIONS) - 1)
    REP_REGIONS[rep] = REGIONS[region_idx]

# ── Seasonality multipliers (by month) ──
SEASONALITY = {
    1: 0.85, 2: 0.80, 3: 0.90, 4: 0.95, 5: 1.00, 6: 1.05,
    7: 1.00, 8: 0.95, 9: 1.05, 10: 1.10, 11: 1.30, 12: 1.45
}


def generate_dates(n):
    total_days = (END_DATE - START_DATE).days
    dates = []
    for _ in range(n):
        month = np.random.choice(range(1, 13), p=[
            s / sum(SEASONALITY.values()) for s in SEASONALITY.values()
        ])
        day = random.randint(1, 28)
        dates.append(datetime(2025, month, day))
    return dates


def generate_sales_data(n):
    dates = generate_dates(n)
    rows = []

    for i in range(n):
        order_id = f"ORD-{100000 + i}"
        date = dates[i]
        region = np.random.choice(REGIONS, p=REGION_WEIGHTS)
        category = random.choice(list(CATEGORIES.keys()))
        product = random.choice(PRODUCTS[category])

        # Pick a rep from this region (or random if mismatch — adds realism)
        region_reps = [r for r, reg in REP_REGIONS.items() if reg == region]
        if region_reps:
            sales_rep = random.choice(region_reps)
        else:
            sales_rep = random.choice(SALES_REPS)

        customer_segment = np.random.choice(CUSTOMER_SEGMENTS, p=SEGMENT_WEIGHTS)

        price_lo, price_hi = CATEGORIES[category]["price_range"]
        unit_price = round(np.random.uniform(price_lo, price_hi), 2)

        # Units: Corporate buys more
        if customer_segment == "Corporate":
            units = np.random.choice(range(1, 25), p=np.array([max(0.01, 0.30 - 0.012 * x) for x in range(24)]) / sum([max(0.01, 0.30 - 0.012 * x) for x in range(24)]))
        elif customer_segment == "Small Business":
            units = random.randint(1, 10)
        else:
            units = random.randint(1, 5)

        # Discount logic
        if customer_segment == "Corporate":
            discount = round(np.random.choice([0, 0.05, 0.10, 0.15, 0.20], p=[0.20, 0.25, 0.30, 0.15, 0.10]), 2)
        elif customer_segment == "Small Business":
            discount = round(np.random.choice([0, 0.05, 0.10], p=[0.40, 0.35, 0.25]), 2)
        else:
            discount = round(np.random.choice([0, 0.05, 0.10, 0.15], p=[0.50, 0.25, 0.15, 0.10]), 2)

        revenue = round(unit_price * units * (1 - discount), 2)

        margin_lo, margin_hi = CATEGORIES[category]["margin_range"]
        cost_pct = 1 - np.random.uniform(margin_lo, margin_hi)
        cogs = round(unit_price * units * cost_pct, 2)

        # Monthly target per rep (~$40K/month baseline, adjusted by seasonality)
        target = round(40000 * SEASONALITY[date.month] / 30, 2)

        rows.append({
            "order_id": order_id,
            "order_date": date.strftime("%Y-%m-%d"),
            "region": region,
            "sales_rep": sales_rep,
            "customer_segment": customer_segment,
            "product_category": category,
            "product_name": product,
            "unit_price": unit_price,
            "units_sold": int(units),
            "discount_pct": discount,
            "revenue": revenue,
            "cogs": cogs,
            "daily_target": target,
        })

    return pd.DataFrame(rows)


def inject_data_quality_issues(df):
    """Inject realistic messiness for the cleaning notebook to address."""
    df = df.copy()
    n = len(df)
    rng = np.random.default_rng(42)

    # 1. ~2% missing revenue values
    missing_rev = rng.choice(n, size=int(n * 0.02), replace=False)
    df.loc[missing_rev, "revenue"] = np.nan

    # 2. ~1.5% missing region
    missing_region = rng.choice(n, size=int(n * 0.015), replace=False)
    df.loc[missing_region, "region"] = np.nan

    # 3. ~1% missing sales_rep
    missing_rep = rng.choice(n, size=int(n * 0.01), replace=False)
    df.loc[missing_rep, "sales_rep"] = np.nan

    # 4. ~80 duplicate rows
    dup_indices = rng.choice(n, size=80, replace=False)
    duplicates = df.iloc[dup_indices].copy()
    df = pd.concat([df, duplicates], ignore_index=True)

    # 5. ~0.5% negative revenue (data entry errors)
    neg_rev = rng.choice(len(df), size=int(len(df) * 0.005), replace=False)
    df.loc[neg_rev, "revenue"] = df.loc[neg_rev, "revenue"].apply(
        lambda x: -abs(x) if pd.notna(x) else x
    )

    # 6. Inconsistent region casing (~1%)
    case_issues = rng.choice(len(df), size=int(len(df) * 0.01), replace=False)
    for idx in case_issues:
        if pd.notna(df.loc[idx, "region"]):
            df.loc[idx, "region"] = df.loc[idx, "region"].upper()

    # 7. A few outlier unit prices (10x normal — fat finger errors)
    outlier_idx = rng.choice(len(df), size=8, replace=False)
    df.loc[outlier_idx, "unit_price"] = df.loc[outlier_idx, "unit_price"] * 10

    # 8. Some dates as different format (~0.5%)
    fmt_issues = rng.choice(len(df), size=int(len(df) * 0.005), replace=False)
    for idx in fmt_issues:
        if pd.notna(df.loc[idx, "order_date"]):
            d = datetime.strptime(df.loc[idx, "order_date"], "%Y-%m-%d")
            df.loc[idx, "order_date"] = d.strftime("%m/%d/%Y")

    # 9. Whitespace in some product names
    ws_issues = rng.choice(len(df), size=int(len(df) * 0.008), replace=False)
    for idx in ws_issues:
        df.loc[idx, "product_name"] = "  " + df.loc[idx, "product_name"] + "  "

    # Shuffle the dataframe
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    return df


def main():
    print("Generating synthetic sales data...")
    df = generate_sales_data(NUM_ROWS)

    print(f"  Clean rows: {len(df)}")
    df = inject_data_quality_issues(df)
    print(f"  After injecting issues: {len(df)} rows")

    os.makedirs("data/raw", exist_ok=True)
    output_path = "data/raw/sales_data_raw.csv"
    df.to_csv(output_path, index=False)
    print(f"  Saved to {output_path}")

    # Quick summary
    print("\n── Data Summary ──")
    print(f"  Date range: 2025-01-01 to 2025-12-31")
    print(f"  Regions: {len(REGIONS)}")
    print(f"  Sales Reps: {len(SALES_REPS)}")
    print(f"  Product Categories: {len(CATEGORIES)}")
    print(f"  Products: {sum(len(v) for v in PRODUCTS.values())}")
    print(f"  Customer Segments: {len(CUSTOMER_SEGMENTS)}")
    print(f"\n  Quality issues injected:")
    print(f"    - ~240 missing revenue values")
    print(f"    - ~180 missing regions")
    print(f"    - ~120 missing sales reps")
    print(f"    - ~80 duplicate rows")
    print(f"    - ~60 negative revenue entries")
    print(f"    - ~120 inconsistent region casing")
    print(f"    - 8 outlier unit prices (10x)")
    print(f"    - ~60 inconsistent date formats")
    print(f"    - ~96 whitespace in product names")


if __name__ == "__main__":
    main()

Generating synthetic sales data...
  Clean rows: 12000
  After injecting issues: 12080 rows
  Saved to data/raw/sales_data_raw.csv

── Data Summary ──
  Date range: 2025-01-01 to 2025-12-31
  Regions: 5
  Sales Reps: 12
  Product Categories: 5
  Products: 30
  Customer Segments: 3

  Quality issues injected:
    - ~240 missing revenue values
    - ~180 missing regions
    - ~120 missing sales reps
    - ~80 duplicate rows
    - ~60 negative revenue entries
    - ~120 inconsistent region casing
    - 8 outlier unit prices (10x)
    - ~60 inconsistent date formats
    - ~96 whitespace in product names


In [29]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [30]:
df = pd.read_csv('data/raw/sales_data_raw.csv')
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)


Shape: 12,080 rows × 13 columns


,order_id,order_date,region,sales_rep,customer_segment,product_category,product_name,unit_price,units_sold,discount_pct,revenue,cogs,daily_target
0,ORD-110288,2025-09-19,Southeast,Sofia Rodriguez,Consumer,Electronics,Portable Charger,530.00,5,0.05,2517.50,2069.45,1400.00
1,ORD-109681,2025-01-13,West,Ryan Mitchell,Corporate,Apparel,Winter Jacket,73.26,24,0.00,1758.24,858.16,1133.33
2,ORD-102901,2025-10-08,Southeast,Sofia Rodriguez,Small Business,Electronics,Tablet,202.10,5,0.05,959.97,874.52,1466.67
3,ORD-111256,2025-01-26,Southwest,Olivia Foster,Corporate,Electronics,Webcam,373.55,16,0.05,5677.96,5203.15,1133.33
4,ORD-108351,2025-01-22,Southwest,Carlos Mendez,Corporate,Sports,Dumbbells,108.45,10,0.05,1030.27,763.70,1133.33
5,ORD-108446,2025-08-05,Southeast,James O'Brien,Consumer,Sports,Dumbbells,162.19,2,0.00,324.38,222.10,1266.67
6,ORD-109495,2025-07-02,Southeast,Sofia Rodriguez,Corporate,Apparel,Baseball Cap,126.29,6,0.00,757.74,427.63,1333.33
7,ORD-111945,2025-11-02,Southwest,Carlos Mendez,Consumer,Office Supplies,Ergonomic Mouse,51.42,2,0.00,102.84,56.50,1733.33
8,ORD-101747,2025-04-28,Southeast,Sofia Rodriguez,Consumer,Sports,Yoga Mat,104.79,3,0.15,267.21,231.98,1266.67
9,ORD-103128,2025-12-26,Northeast,Marcus Thompson,Consumer,Apparel,Polo Shirt,169.84,5,0.00,849.20,382.79,1933.33


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12080 entries, 0 to 12079
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          12080 non-null  object 
 1   order_date        12080 non-null  object 
 2   region            11899 non-null  object 
 3   sales_rep         11959 non-null  object 
 4   customer_segment  12080 non-null  object 
 5   product_category  12080 non-null  object 
 6   product_name      12080 non-null  object 
 7   unit_price        12080 non-null  float64
 8   units_sold        12080 non-null  int64  
 9   discount_pct      12080 non-null  float64
 10  revenue           11839 non-null  float64
 11  cogs              12080 non-null  float64
 12  daily_target      12080 non-null  float64
dtypes: float64(5), int64(1), object(7)
memory usage: 1.2+ MB


In [32]:
df.describe()

,unit_price,units_sold,discount_pct,revenue,cogs,daily_target
count,12080.00,12080.00,12080.00,11839.00,12080.00,12080.00
mean,230.75,5.27,0.06,1120.10,879.40,1414.68
std,220.49,4.49,0.06,1685.74,1478.91,250.81
min,5.07,1.00,0.00,-6402.17,2.79,1066.67
25%,82.05,2.00,0.00,231.14,148.27,1266.67
50%,150.74,4.00,0.05,547.29,365.43,1333.33
75%,309.30,7.00,0.10,1289.88,945.64,1466.67
max,3290.40,24.00,0.20,21456.48,19380.55,1933.33


In [33]:
quality_report = pd.DataFrame({
    'dtypes' : df.dtypes,
    'null' : df.isnull().sum(),
    'null.pct' : (df.isnull().sum() / len(df)* 100).round(2),
    'unique' : df.nunique(),
     'sample_value' : df.iloc[0]
})
print (quality_report)


                   dtypes  null  null.pct  unique      sample_value
order_id           object     0      0.00   12000        ORD-110288
order_date         object     0      0.00     392        2025-09-19
region             object   181      1.50      10         Southeast
sales_rep          object   121      1.00      12   Sofia Rodriguez
customer_segment   object     0      0.00       3          Consumer
product_category   object     0      0.00       5       Electronics
product_name       object     0      0.00      59  Portable Charger
unit_price        float64     0      0.00   10509            530.00
units_sold          int64     0      0.00      24                 5
discount_pct      float64     0      0.00       5              0.05
revenue           float64   241      2.00   11336           2517.50
cogs              float64     0      0.00   11379           2069.45
daily_target      float64     0      0.00       9           1400.00


In [34]:
dup_count = df.duplicated(subset='order_id', keep='first').sum()
dup_ids = df[df.duplicated(subset='order_id', keep=False)]['order_id'].unique()
df[df['order_id'].isin(dup_ids)].sort_values('order_id').head(6)

,order_id,order_date,region,sales_rep,customer_segment,product_category,product_name,unit_price,units_sold,discount_pct,revenue,cogs,daily_target
5546,ORD-100166,2025-05-06,Midwest,Aisha Patel,Consumer,Home & Garden,Air Purifier,51.13,5,0.00,255.65,175.84,1333.33
8432,ORD-100166,2025-05-06,Midwest,Aisha Patel,Consumer,Home & Garden,Air Purifier,51.13,5,0.00,255.65,175.84,1333.33
11237,ORD-100492,2025-04-22,West,Chen Wei,Corporate,Office Supplies,Ergonomic Mouse,96.49,8,0.10,694.73,464.92,1266.67
8597,ORD-100492,2025-04-22,West,Chen Wei,Corporate,Office Supplies,Ergonomic Mouse,96.49,8,0.10,-694.73,464.92,1266.67
87,ORD-100533,2025-08-05,Northeast,Marcus Thompson,Corporate,Office Supplies,Pen Set,105.39,10,0.15,895.82,618.64,1266.67
491,ORD-100533,2025-08-05,Northeast,Marcus Thompson,Corporate,Office Supplies,Pen Set,105.39,10,0.15,895.82,618.64,1266.67


In [35]:
df.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)

,order_id,order_date,region,sales_rep,customer_segment,product_category,product_name,unit_price,units_sold,discount_pct,revenue,cogs,daily_target
0,ORD-110288,2025-09-19,Southeast,Sofia Rodriguez,Consumer,Electronics,Portable Charger,530.00,5,0.05,2517.50,2069.45,1400.00
1,ORD-109681,2025-01-13,West,Ryan Mitchell,Corporate,Apparel,Winter Jacket,73.26,24,0.00,1758.24,858.16,1133.33
2,ORD-102901,2025-10-08,Southeast,Sofia Rodriguez,Small Business,Electronics,Tablet,202.10,5,0.05,959.97,874.52,1466.67
3,ORD-111256,2025-01-26,Southwest,Olivia Foster,Corporate,Electronics,Webcam,373.55,16,0.05,5677.96,5203.15,1133.33
4,ORD-108351,2025-01-22,Southwest,Carlos Mendez,Corporate,Sports,Dumbbells,108.45,10,0.05,1030.27,763.70,1133.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,ORD-111964,2025-08-13,Southwest,Kwame Asante,Consumer,Apparel,Sunglasses,19.14,3,0.00,57.42,26.41,1266.67
11996,ORD-105191,2025-10-23,West,Chen Wei,Corporate,Sports,Water Bottle,323.75,15,0.10,4370.62,3579.47,1466.67
11997,ORD-105390,03/22/2025,Southeast,Sofia Rodriguez,Consumer,Electronics,Bluetooth Speaker,361.34,4,0.10,1300.82,1307.61,1200.00
11998,ORD-100860,2025-10-22,Northeast,Marcus Thompson,Corporate,Home & Garden,Tool Kit,352.30,9,0.10,2853.63,1859.38,1466.67


In [36]:
rows_before = len(df)
df = df.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)
rows_after = len(df)
print(f"Removed {rows_before - rows_after} duplicates → {rows_after:,} rows remaining")

Removed 80 duplicates → 12,000 rows remaining


In [37]:
print("Missing values before treatment:")
print(df.isnull().sum()[df.isnull().sum()>0])


Missing values before treatment:
region       180
sales_rep    120
revenue      240
dtype: int64


In [38]:
missing_rev_mask = df['revenue'].isnull()
df.loc[missing_rev_mask, 'revenue'] = (
    df.loc[missing_rev_mask, 'unit_price'] *
    df.loc[missing_rev_mask, 'units_sold'] *
    (1-df.loc[missing_rev_mask, 'discount_pct'])
).round(2)

In [39]:
print(f"Recalculated {missing_rev_mask.sum()} revenue values")
print(f"Remaining null revenue: {df['revenue'].isnull().sum()}")

Recalculated 240 revenue values
Remaining null revenue: 0


In [40]:
# 3b. Infer missing regions from sales rep assignments
# Build a rep-to-region lookup from non-null rows
rep_region_map = (
    df.dropna(subset=['sales_rep', 'region'])
    .groupby('sales_rep')['region']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    .to_dict()
)

missing_region_mask = df['region'].isnull()
df.loc[missing_region_mask, 'region'] = df.loc[missing_region_mask, 'sales_rep'].map(rep_region_map)

# Any still missing → "Unknown"
still_missing = df['region'].isnull().sum()
df['region'] = df['region'].fillna('Unknown')
print(f"Inferred {missing_region_mask.sum() - still_missing} regions from rep lookup")
print(f"Flagged {still_missing} as 'Unknown'")

Inferred 177 regions from rep lookup
Flagged 3 as 'Unknown'


In [41]:
df['sales_rep'] = df['sales_rep'].fillna('Unknown')
print(f"Flagged {df['sales_rep'].isnull().sum()} as 'Unknown'")

Flagged 0 as 'Unknown'


In [42]:
# Verify: zero nulls remaining
print("Missing values after treatment:")
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("✓ All missing values resolved")

Missing values after treatment:
Series([], dtype: int64)
✓ All missing values resolved


In [43]:
slash_dates = df['order_date'].str.contains('/').sum()
dash_dates = df['order_date'].str.contains('-').sum()
print(f"Found {slash_dates} / dates and {dash_dates} - dates")

Found 60 / dates and 11940 - dates


In [44]:
df['order_date'] = pd.to_datetime(df['order_date'], format="mixed", dayfirst=False)
print(df['order_date'])

0       2025-09-19
1       2025-01-13
2       2025-10-08
3       2025-01-26
4       2025-01-22
           ...    
11995   2025-08-13
11996   2025-10-23
11997   2025-03-22
11998   2025-10-22
11999   2025-05-17
Name: order_date, Length: 12000, dtype: datetime64[ns]


In [45]:
# Ensure numeric columns are correct type
numeric_cols = ['unit_price', 'units_sold', 'discount_pct', 'revenue', 'cogs', 'daily_target']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['units_sold'] = df['units_sold'].astype(int)
print("Numeric columns confirmed:")
print(df[numeric_cols].dtypes)

Numeric columns confirmed:
unit_price      float64
units_sold        int64
discount_pct    float64
revenue         float64
cogs            float64
daily_target    float64
dtype: object


In [46]:
# Check region values before
print("Region values (before):")
print(df['region'].value_counts())

Region values (before):
region
Northeast    2983
West         2590
Southeast    2458
Midwest      2344
Southwest    1505
SOUTHEAST      28
NORTHEAST      26
MIDWEST        25
SOUTHWEST      19
WEST           19
Unknown         3
Name: count, dtype: int64


In [47]:
df['region'] = df['region'].str.strip().str.title()

print("Region values (after):")
print(df['region'].value_counts())

Region values (after):
region
Northeast    3009
West         2609
Southeast    2486
Midwest      2369
Southwest    1524
Unknown         3
Name: count, dtype: int64


In [48]:
string_cols = df.select_dtypes(include='object').columns
for col in string_cols:
    df[col] =df[col].str.strip()

print(f"Unique products: {df['product_name'].nunique()}")
print(df['product_name'].value_counts().head(10))


Unique products: 30
product_name
Yoga Mat          445
Polo Shirt        441
Jump Rope         440
Desk Organizer    431
Pen Set           427
Candle Set        413
Whiteboard        411
Planner           410
Water Bottle      404
Smart Watch       404
Name: count, dtype: int64


In [49]:
neg_rev = df[df['revenue']<0]
print(f"Negative revenue rows: {len(neg_rev)}")
neg_rev[['order_id', 'product_category', 'unit_price', 'units_sold', 'discount_pct', 'revenue' ]].head()

Negative revenue rows: 59


,order_id,product_category,unit_price,units_sold,discount_pct,revenue
76,ORD-111488,Sports,252.27,9,0.00,-2270.43
107,ORD-108142,Apparel,50.24,4,0.10,-180.86
145,ORD-111768,Office Supplies,42.17,23,0.05,-921.41
442,ORD-106688,Sports,195.83,2,0.00,-391.66
711,ORD-109660,Sports,175.33,5,0.00,-876.65


In [50]:
neg_mask = df['revenue']<0
df.loc[neg_mask, 'revenue'] = (
    df.loc[neg_mask, 'unit_price']
    * df.loc[neg_mask, 'units_sold']
    * (1 - df.loc[neg_mask, 'discount_pct'])
).round(2)
print(f'Fixed {neg_mask.sum()} negative revenue entries')
print(f'Remaining negative: {(df['revenue']<0).sum()}')

Fixed 59 negative revenue entries
Remaining negative: 0


In [51]:
price_stats = df.groupby('product_category')['unit_price'].agg(['min', 'max', 'mean', 'median', 'std'])
price_stats['threshold'] = price_stats['mean'] + 3 * price_stats['std']
pass

In [52]:
print(price_stats)

                   min     max   mean  median    std  threshold
product_category                                               
Apparel          15.19  339.80 108.20  107.94  53.58     268.94
Electronics      50.61  999.84 526.19  523.71 275.55    1352.83
Home & Garden    10.04 2995.90 260.41  263.07 153.50     720.90
Office Supplies   5.07 1254.80  79.69   80.50  51.12     233.05
Sports           19.99 3290.40 187.63  187.25 114.14     530.05


In [53]:
outlier_count = 0
for category in df['product_category'].unique():
    cat_mask = df['product_category'] == category
    p99 = df.loc[cat_mask, 'unit_price'].quantile(0.99)
    outlier_mask = cat_mask & (df['unit_price'] > p99 * 2)
    count = outlier_mask.sum()
    if count > 0:
       df.loc[outlier_mask, 'unit_price'] = round(99, 2)
       df.loc[outlier_mask, 'revenue'] = (
           df.loc[outlier_mask, 'unit_price']
           * df.loc[outlier_mask, 'units_sold']
           * (1 - df.loc[outlier_mask, 'discount_pct'])
       ).round(2)
       outlier_count += count
print(f"\nTotal outlier prices corrected: {outlier_count}")



Total outlier prices corrected: 6


In [54]:
# Validation checks
checks = {
    'No null values': df.isnull().sum().sum() == 0,
    'No duplicate order IDs': df['order_id'].is_unique,
    'All revenue >= 0': (df['revenue'] >= 0).all(),
    'All units_sold > 0': (df['units_sold'] > 0).all(),
    'Discount between 0 and 1': ((df['discount_pct'] >= 0) & (df['discount_pct'] <= 1)).all(),
    'COGS > 0': (df['cogs'] > 0).all(),
    'Dates within 2025': (df['order_date'].dt.year == 2025).all(),
    'All regions valid': set(df['region'].unique()).issubset(
        {'Northeast', 'Southeast', 'Midwest', 'West', 'Southwest', 'Unknown'}
    ),
}

for check, passed in checks.items():
    status = '✓' if passed else '✗'
    print(f"  {status} {check}")

  ✓ No null values
  ✓ No duplicate order IDs
  ✓ All revenue >= 0
  ✓ All units_sold > 0
  ✓ Discount between 0 and 1
  ✓ COGS > 0
  ✓ Dates within 2025
  ✓ All regions valid


In [55]:
#Time-based features
df['order_month'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.strftime('%b')
df['order_quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str)
df['order_week'] = df['order_date'].dt.isocalendar().week.astype(int)
df['order_dayofweek'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

In [56]:
df['gross_profit'] =  (df['revenue'] - df['cogs']).round(2)
df['gross_margin_pct'] = (df['gross_profit'] / df['revenue']).round(4)
df['revenue_per_month'] = (df['revenue'] / df['units_sold']).round(2)
df['discount_amount'] = (df['unit_price'] * df['units_sold'] * df['discount_pct']).round(2)


In [57]:
df['order_tier'] = pd.cut(
    df['revenue'],
    bins=[0, 50, 200, 500, 1500, float('inf')],
    labels=['Micro (<$50)', 'Small ($50-$200)', 'Medium ($200-500)', 'Large ($500-1500)', 'Enterprise ($1500+)']
)

print('Order tier distribution')
print(df['order_tier'].value_counts().sort_index())

Order tier distribution
order_tier
Micro (<$50)            412
Small ($50-$200)       2171
Medium ($200-500)      3041
Large ($500-1500)      3792
Enterprise ($1500+)    2584
Name: count, dtype: int64


In [58]:
df['margin_health'] = pd.cut(
    df['gross_margin_pct'],
    bins=[-float('inf'), 0.15, 0.25, 0.40, float('inf')],
    labels=['Critical', 'Low', 'Healthy', 'Strong']
)

print('Margin health distribution')
print(df['margin_health'].value_counts())

Margin health distribution
margin_health
Healthy     4868
Strong      3003
Low         2280
Critical    1849
Name: count, dtype: int64


In [59]:
df['target_achievement'] = (df['revenue'] / df['daily_target']).round(4)
df['above_target'] = df['revenue'] > df['daily_target']

In [60]:
# Final column check
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nNew columns added: {df.shape[1] - 13}")
print(f"Columns: {list(df.columns)}")

Final shape: 12,000 rows × 27 columns

New columns added: 14
Columns: ['order_id', 'order_date', 'region', 'sales_rep', 'customer_segment', 'product_category', 'product_name', 'unit_price', 'units_sold', 'discount_pct', 'revenue', 'cogs', 'daily_target', 'order_month', 'order_month_name', 'order_quarter', 'order_week', 'order_dayofweek', 'is_weekend', 'gross_profit', 'gross_margin_pct', 'revenue_per_month', 'discount_amount', 'order_tier', 'margin_health', 'target_achievement', 'above_target']


In [61]:
df.head()

,order_id,order_date,region,sales_rep,customer_segment,product_category,product_name,unit_price,units_sold,discount_pct,revenue,cogs,daily_target,order_month,order_month_name,order_quarter,order_week,order_dayofweek,is_weekend,gross_profit,gross_margin_pct,revenue_per_month,discount_amount,order_tier,margin_health,target_achievement,above_target
0,ORD-110288,2025-09-19,Southeast,Sofia Rodriguez,Consumer,Electronics,Portable Charger,530.00,5,0.05,2517.50,2069.45,1400.00,9,Sep,Q3,38,Friday,False,448.05,0.18,503.50,132.50,Enterprise ($1500+),Low,1.80,True
1,ORD-109681,2025-01-13,West,Ryan Mitchell,Corporate,Apparel,Winter Jacket,73.26,24,0.00,1758.24,858.16,1133.33,1,Jan,Q1,3,Monday,False,900.08,0.51,73.26,0.00,Enterprise ($1500+),Strong,1.55,True
2,ORD-102901,2025-10-08,Southeast,Sofia Rodriguez,Small Business,Electronics,Tablet,202.10,5,0.05,959.97,874.52,1466.67,10,Oct,Q4,41,Wednesday,False,85.45,0.09,191.99,50.53,Large ($500-1500),Critical,0.65,False
3,ORD-111256,2025-01-26,Southwest,Olivia Foster,Corporate,Electronics,Webcam,373.55,16,0.05,5677.96,5203.15,1133.33,1,Jan,Q1,4,Sunday,True,474.81,0.08,354.87,298.84,Enterprise ($1500+),Critical,5.01,True
4,ORD-108351,2025-01-22,Southwest,Carlos Mendez,Corporate,Sports,Dumbbells,108.45,10,0.05,1030.27,763.70,1133.33,1,Jan,Q1,4,Wednesday,False,266.57,0.26,103.03,54.22,Large ($500-1500),Healthy,0.91,False


In [62]:
# Final summary stats
print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"  Original rows:        12,080")
print(f"  Final rows:           {len(df):,}")
print(f"  Duplicates removed:   ~80")
print(f"  Nulls resolved:       ~540")
print(f"  Negative revenue fixed: ~60")
print(f"  Outlier prices capped:  ~8")
print(f"  Date formats unified:   ~60")
print(f"  Casing standardized:    ~120")
print(f"  Features engineered:    {df.shape[1] - 13}")
print(f"\n  Revenue range: ${df['revenue'].min():,.2f} – ${df['revenue'].max():,.2f}")
print(f"  Total revenue: ${df['revenue'].sum():,.2f}")
print(f"  Avg margin:    {df['gross_margin_pct'].mean():.1%}")

CLEANING SUMMARY
  Original rows:        12,080
  Final rows:           12,000
  Duplicates removed:   ~80
  Nulls resolved:       ~540
  Negative revenue fixed: ~60
  Outlier prices capped:  ~8
  Date formats unified:   ~60
  Casing standardized:    ~120
  Features engineered:    14

  Revenue range: $5.43 – $21,456.48
  Total revenue: $13,572,415.49
  Avg margin:    30.2%


In [63]:
# Export
import os
os.makedirs('data/cleaned', exist_ok=True)
df.to_csv('data/cleaned/sales_data_cleaned.csv', index=False)
print(f"✓ Exported to data/cleaned/sales_data_cleaned.csv")
print(f"  File size: {os.path.getsize('data/cleaned/sales_data_cleaned.csv') / 1024:.0f} KB")

✓ Exported to data/cleaned/sales_data_cleaned.csv
  File size: 2366 KB


In [64]:
# Code to see and download sales_data_cleaned.csv
print("\n--- Viewing sales_data_cleaned.csv ---")
cleaned_df = pd.read_csv('data/cleaned/sales_data_cleaned.csv')
print(f"Shape of cleaned data: {cleaned_df.shape[0]:,} rows × {cleaned_df.shape[1]} columns")
print("First 5 rows of cleaned data:")
print(cleaned_df.head())

print("\n--- Downloading sales_data_cleaned.csv ---")
from google.colab import files
try:
    files.download('data/cleaned/sales_data_cleaned.csv')
    print("Download initiated for sales_data_cleaned.csv.")
except Exception as e:
    print(f"Error during download: {e}")
    print("Please ensure you run this cell in a Colab environment to enable file download.")


--- Viewing sales_data_cleaned.csv ---
Shape of cleaned data: 12,000 rows × 27 columns
First 5 rows of cleaned data:
     order_id  order_date     region        sales_rep customer_segment  \
0  ORD-110288  2025-09-19  Southeast  Sofia Rodriguez         Consumer   
1  ORD-109681  2025-01-13       West    Ryan Mitchell        Corporate   
2  ORD-102901  2025-10-08  Southeast  Sofia Rodriguez   Small Business   
3  ORD-111256  2025-01-26  Southwest    Olivia Foster        Corporate   
4  ORD-108351  2025-01-22  Southwest    Carlos Mendez        Corporate   

  product_category      product_name  unit_price  units_sold  discount_pct  \
0      Electronics  Portable Charger      530.00           5          0.05   
1          Apparel     Winter Jacket       73.26          24          0.00   
2      Electronics            Tablet      202.10           5          0.05   
3      Electronics            Webcam      373.55          16          0.05   
4           Sports         Dumbbells      108.4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated for sales_data_cleaned.csv.
